In [1]:
import sys, torch
print(sys.executable)
print(torch.__version__)

/home1/jtt_1/mtp/python-env-1080ti/bin/python
2.7.1+cu118


In [2]:
import numpy as np
if not hasattr(np, "int"):
    np.int = int


In [3]:
import os, sys

# Path to your Video-Swin-Transformer project
ROOT = "/home1/jtt_1/mtp/Video-Swin-Transformer"

# Add project and mmaction_local to Python path
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "mmaction_local"))

# ✅ Verify
print("[INFO] Added to sys.path:")
for p in sys.path[:3]:
    print("  ", p)

# Now safe to import
from mmaction.models import BACKBONES
from mmaction_local.models.backbones.swin_transformer3d import SwinTransformer3D

# Check registration
if "SwinTransformer3D" not in BACKBONES.module_dict:
    BACKBONES.register_module(SwinTransformer3D)
    print("✅ Registered SwinTransformer3D in BACKBONES registry.")
else:
    print("ℹ️ SwinTransformer3D already registered.")


[INFO] Added to sys.path:
   /home1/jtt_1/mtp/Video-Swin-Transformer/mmaction_local
   /home1/jtt_1/mtp/Video-Swin-Transformer
   /apps/anaconda3/lib/python310.zip


/home1/jtt_1/mtp/python-env-1080ti/lib/python3.10/site-packages/mmcv/cnn/bricks/transformer.py:28: UserWarning: Fail to import ``MultiScaleDeformableAttention`` from ``mmcv.ops.multi_scale_deform_attn``, You should install ``mmcv-full`` if you need this module. 
  warnings.warn('Fail to import ``MultiScaleDeformableAttention`` from '


ℹ️ SwinTransformer3D already registered.


In [4]:
import sys, os

# ============================================================
# 🧩 Add paths so notebook can find local MMACTION + configs
# ============================================================
sys.path.insert(0, "/home1/jtt_1/mtp/Video-Swin-Transformer")
sys.path.insert(0, "/home1/jtt_1/mtp/Video-Swin-Transformer/mmaction_local")

# ============================================================
# ✅ Verify imports
# ============================================================
try:
    from mmaction.apis import init_recognizer, inference_recognizer
    from ultralytics import YOLO
    import torch

    print("[INFO] ✅ All modules imported successfully")
    print("[INFO] PyTorch version:", torch.__version__)
    print("[INFO] CUDA available:", torch.cuda.is_available())
except Exception as e:
    print("❌ Import failed:", e)


[INFO] ✅ All modules imported successfully
[INFO] PyTorch version: 2.7.1+cu118
[INFO] CUDA available: True


In [5]:
import os
import time
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from collections import deque
from ultralytics import YOLO
import pandas as pd
import csv

import os
import torch

# ============================================================
# ⚙️ CONFIGURATION (Cleaned)
# ============================================================
CONFIG = {
    # 🎞 Input video
    "VIDEO_PATH": "/home1/jtt_1/mtp/okutama-action/TestSetVideos/Drone2/Noon/2.2.1.mp4",

    # 🧠 Model paths
    "YOLO_MODEL_PATH": "/home1/jtt_1/mtp/models/yolo-e30.pt",
    "TSF_BASE_MODEL": "/home1/jtt_1/mtp/timesformer_base_patch16_224_k400",
    "TSF_SCENE_CKPT": "/home1/jtt_1/mtp/models/best_timesformer_epoch2.pt",

    # 🗂 Dataset CSV (for label decoding)
    "TRAIN_CSV": "/home1/jtt_1/mtp/timesformer-dataset-train/train_labels.csv",

    # 🔧 Inference parameters
    "CONF_THRESH": 0.7,             # YOLO detection threshold
    "DETECT_EVERY_N_FRAMES": 1,     # Run YOLO every N frames
    "NUM_FRAMES": 4,                # Frames per clip for TimeSformer
    "TOP_K": 2,                     # Number of top predicted actions
    "OUT_OF_FRAME_FRACTION": 0.6,   # jTracker re-detect tolerance
    "TEST_MODE": True,              # Limit frames for quick testing
    "TEST_MAX_FRAMES": 100,         # Max frames in test mode

    # ⚙️ System configuration
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",

    "REID_MEMORY_TIME": 10.0,        # seconds to keep ReID features
}

# ============================================================
# 🧩 AUTO-DERIVED PATHS
# ============================================================
CONFIG["VIDEO_NAME"] = os.path.splitext(os.path.basename(CONFIG["VIDEO_PATH"]))[0]
CONFIG["OUTPUT_DIR"] = f"/home1/jtt_1/mtp/outputs/{CONFIG['VIDEO_NAME']}"
CONFIG["RESULTS_CSV"] = os.path.join(CONFIG["OUTPUT_DIR"], f"{CONFIG['VIDEO_NAME']}_tracking.csv")
os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

# ============================================================
# 📋 INFO LOGGING (Simplified)
# ============================================================
print("=" * 80)
print(f"🎥  VIDEO NAME           : {CONFIG['VIDEO_NAME']}")
print(f"📂  OUTPUT DIRECTORY      : {CONFIG['OUTPUT_DIR']}")
print("-" * 80)
print(f"🧠  YOLO DETECTOR         : {CONFIG['YOLO_MODEL_PATH']}")
print(f"🟦  TimeSformer (Scene)   : {CONFIG['TSF_SCENE_CKPT']}")
print(f"🧾  LABELS CSV            : {CONFIG['TRAIN_CSV']}")
print("-" * 80)
print(f"⚙️  DEVICE                : {CONFIG['DEVICE']}")
print(f"🎬  YOLO CONF THRESHOLD   : {CONFIG['CONF_THRESH']}")
print(f"📸  DETECT EVERY N FRAMES : {CONFIG['DETECT_EVERY_N_FRAMES']}")
print(f"🎞️  FRAMES PER CLIP       : {CONFIG['NUM_FRAMES']}")
print(f"🏆  TOP-K ACTIONS          : {CONFIG['TOP_K']}")
print("-" * 80)
if CONFIG["TEST_MODE"]:
    print(f"⚡ TEST MODE ENABLED → using first {CONFIG['TEST_MAX_FRAMES']} frames only")
else:
    print("✅ Full-length video processing enabled")
print("=" * 80)


🎥  VIDEO NAME           : 2.2.1
📂  OUTPUT DIRECTORY      : /home1/jtt_1/mtp/outputs/2.2.1
--------------------------------------------------------------------------------
🧠  YOLO DETECTOR         : /home1/jtt_1/mtp/models/yolo-e30.pt
🟦  TimeSformer (Scene)   : /home1/jtt_1/mtp/models/best_timesformer_epoch2.pt
🧾  LABELS CSV            : /home1/jtt_1/mtp/timesformer-dataset-train/train_labels.csv
--------------------------------------------------------------------------------
⚙️  DEVICE                : cuda
🎬  YOLO CONF THRESHOLD   : 0.7
📸  DETECT EVERY N FRAMES : 1
🎞️  FRAMES PER CLIP       : 4
🏆  TOP-K ACTIONS          : 2
--------------------------------------------------------------------------------
⚡ TEST MODE ENABLED → using first 100 frames only


In [6]:
import cv2

# ------------------------------------------------------------
# 🎥 VIDEO INFO EXTRACTION
# ------------------------------------------------------------
print("\n[INFO] Checking video properties...")

cap = cv2.VideoCapture(CONFIG["VIDEO_PATH"])

if not cap.isOpened():
    raise FileNotFoundError(f"❌ Could not open video file: {CONFIG['VIDEO_PATH']}")

# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration_sec = total_frames / fps if fps > 0 else 0

# Apply test mode truncation
if CONFIG["TEST_MODE"]:
    total_frames = min(total_frames, CONFIG["TEST_MAX_FRAMES"])

# Print nicely formatted info
print("=" * 60)
print(f"🎞️  Video Name     : {CONFIG['VIDEO_NAME']}")
print(f"📏 Resolution      : {width}x{height}")
print(f"🎬 Total Frames    : {total_frames}")
print(f"⏱️  FPS             : {fps:.2f}")
print(f"🕒 Duration (secs) : {duration_sec:.2f}")
print(f"🕕 Duration (mins) : {duration_sec/60:.2f}")
if CONFIG["TEST_MODE"]:
    print(f"⚡ TEST MODE ON → using first {CONFIG['TEST_MAX_FRAMES']} frames only")
print("=" * 60)

cap.release()



[INFO] Checking video properties...
🎞️  Video Name     : 2.2.1
📏 Resolution      : 3840x2160
🎬 Total Frames    : 100
⏱️  FPS             : 29.97
🕒 Duration (secs) : 45.95
🕕 Duration (mins) : 0.77
⚡ TEST MODE ON → using first 100 frames only


In [7]:
from mmaction.apis import init_recognizer
from ultralytics import YOLO
from transformers import AutoImageProcessor, TimesformerForVideoClassification
import torch

print("[INFO] 🔧 Loading YOLO, TimeSformer (scene), and Swin (person) models...")

# ============================================================
# 🟦 Scene-level model (TimeSformer)
# ============================================================
print("[INFO] Loading TimeSformer model (Scene-level)...")

# 1️⃣ Preprocessor for TimeSformer (same normalization as training)
processor_scene = AutoImageProcessor.from_pretrained(CONFIG["TSF_BASE_MODEL"])

# 2️⃣ Load pretrained backbone (Kinetics-400 base)
scene_model = TimesformerForVideoClassification.from_pretrained(CONFIG["TSF_BASE_MODEL"])

# 3️⃣ Load trained weights (from checkpoint)
state_dict = torch.load(CONFIG["TSF_SCENE_CKPT"], map_location=CONFIG["DEVICE"])

# 4️⃣ Filter out incompatible classifier weights (e.g., 400 vs 12 classes)
filtered_dict = {k: v for k, v in state_dict.items() if "classifier" not in k}

# 5️⃣ Load compatible weights only
load_result = scene_model.load_state_dict(filtered_dict, strict=False)
print("[INFO] ✅ Scene model backbone loaded successfully.")
print(f"       Ignored classifier mismatch (12 vs 400 classes)")
print(f"       Missing keys: {len(load_result.missing_keys)} | Unexpected keys: {len(load_result.unexpected_keys)}")

# 6️⃣ Replace classifier with your 12-class output layer
num_features = scene_model.classifier.in_features
scene_model.classifier = torch.nn.Linear(num_features, 12)

# 7️⃣ Move model to device + eval mode
scene_model.to(CONFIG["DEVICE"])
scene_model.eval()
print("[INFO] ✅ Scene-level TimeSformer model ready for inference.")
print("[INFO] 🧩 Processor (AutoImageProcessor) initialized as `processor_scene`")

# ============================================================
# 🟩 Swin Transformer (person-level)
# ============================================================
CONFIG.update({
    "SWIN_CFG": "/home1/jtt_1/mtp/Video-Swin-Transformer/configs/recognition/swin/swin_tiny_patch244_window877_okutama_multilabel_optimized.py",
    "SWIN_CKPT": "/home1/jtt_1/mtp/models/best_mAP_epoch_23.pth",
})

print("[INFO] Loading Swin Transformer (person-level)...")
swin_model = init_recognizer(CONFIG["SWIN_CFG"], CONFIG["SWIN_CKPT"], device=CONFIG["DEVICE"])
swin_model.eval()
print("[INFO] ✅ SwinTransformer3D model loaded successfully.")

# ============================================================
# 🟨 YOLO (for detection)
# ============================================================
print("[INFO] Loading YOLOv8 detector...")
yolo_model = YOLO(CONFIG["YOLO_MODEL_PATH"])
print("[INFO] ✅ YOLOv8 detector loaded successfully.")

# ============================================================
# ✅ Summary
# ============================================================
print("\n✅ All models loaded successfully!")
print(f"🟦 Scene-level model : TimeSformer ({CONFIG['TSF_SCENE_CKPT']})")
print(f"🟩 Person-level model: SwinTransformer3D ({CONFIG['SWIN_CKPT']})")
print(f"🧠 YOLO model        : {CONFIG['YOLO_MODEL_PATH']}")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[INFO] 🔧 Loading YOLO, TimeSformer (scene), and Swin (person) models...
[INFO] Loading TimeSformer model (Scene-level)...
[INFO] ✅ Scene model backbone loaded successfully.
       Ignored classifier mismatch (12 vs 400 classes)
       Missing keys: 2 | Unexpected keys: 0
[INFO] ✅ Scene-level TimeSformer model ready for inference.
[INFO] 🧩 Processor (AutoImageProcessor) initialized as `processor_scene`
[INFO] Loading Swin Transformer (person-level)...
✅ Patched I3DHead.loss(): top_k_accuracy disabled for multi-label BCE.


/home1/jtt_1/mtp/python-env-1080ti/lib/python3.10/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4314.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


load checkpoint from local path: /home1/jtt_1/mtp/models/best_mAP_epoch_23.pth
[INFO] ✅ SwinTransformer3D model loaded successfully.
[INFO] Loading YOLOv8 detector...
[INFO] ✅ YOLOv8 detector loaded successfully.

✅ All models loaded successfully!
🟦 Scene-level model : TimeSformer (/home1/jtt_1/mtp/models/best_timesformer_epoch2.pt)
🟩 Person-level model: SwinTransformer3D (/home1/jtt_1/mtp/models/best_mAP_epoch_23.pth)
🧠 YOLO model        : /home1/jtt_1/mtp/models/yolo-e30.pt


In [8]:
# ------------------------------------------------------------
# 🎥 VIDEO SETUP
# ------------------------------------------------------------
cap = cv2.VideoCapture(CONFIG["VIDEO_PATH"])
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if CONFIG["TEST_MODE"]:
    total_frames = min(CONFIG["TEST_MAX_FRAMES"], total_frames)
    print(f"[TEST MODE] Limiting to {total_frames} frames")

print(f"[INFO] FPS={fps:.2f}, size={width}x{height}, total_frames={total_frames}")

[TEST MODE] Limiting to 100 frames
[INFO] FPS=29.97, size=3840x2160, total_frames=100


In [9]:
from scipy.optimize import linear_sum_assignment
import cv2
import time

# ----------------------
# Helper functions
# ----------------------
def bbox_to_xywh(b):
    x1, y1, x2, y2 = b
    return [(x1 + x2) / 2.0, (y1 + y2) / 2.0, x2 - x1, y2 - y1]

def clamp(v, a, b):
    return max(a, min(b, v))

def crop_box(image, bbox, pad=0):
    h, w = image.shape[:2]
    x1, y1, x2, y2 = int(bbox[0]) - pad, int(bbox[1]) - pad, int(bbox[2]) + pad, int(bbox[3]) + pad
    x1, y1 = clamp(x1, 0, w-1), clamp(y1, 0, h-1)
    x2, y2 = clamp(x2, 0, w-1), clamp(y2, 0, h-1)
    if x2 <= x1 or y2 <= y1:
        return None
    return image[y1:y2, x1:x2]

def color_histogram_feature(image, bbox, size=(32, 32), bins=(16, 16, 8)):
    """Compute a normalized color histogram (HSV) for the bbox region."""
    crop = crop_box(image, bbox)
    if crop is None or crop.size == 0:
        # fallback: small zero vector
        return np.zeros(bins[0] + bins[1] + bins[2], dtype=np.float32)
    crop = cv2.resize(crop, size, interpolation=cv2.INTER_LINEAR)
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    h_bins, s_bins, v_bins = bins
    hist_h = cv2.calcHist([hsv], [0], None, [h_bins], [0, 180]).flatten()
    hist_s = cv2.calcHist([hsv], [1], None, [s_bins], [0, 256]).flatten()
    hist_v = cv2.calcHist([hsv], [2], None, [v_bins], [0, 256]).flatten()
    hist = np.concatenate([hist_h, hist_s, hist_v]).astype(np.float32)
    if hist.sum() > 0:
        hist /= hist.sum()
    return hist

def cosine_sim(a, b):
    if a is None or b is None or a.sum()==0 or b.sum()==0:
        return 0.0
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


# ----------------------
# Particle filter class (unchanged core but exposed for predict/estimate)
# ----------------------
class Particle:
    def __init__(self, bbox, num_particles=50):
        self.num_particles = num_particles
        self.particles = self.initialize_particles(bbox)
        self.weights = np.ones(num_particles) / num_particles

    def initialize_particles(self, bbox):
        x1, y1, x2, y2 = bbox
        w, h = max(1, x2 - x1), max(1, y2 - y1)
        p = np.zeros((self.num_particles, 4))
        # sample around the box corners rather than absolute box for stability
        p[:, 0] = np.random.normal((x1 + x2) / 2.0 - w * 0.5, w * 0.05, self.num_particles)
        p[:, 1] = np.random.normal((y1 + y2) / 2.0 - h * 0.5, h * 0.05, self.num_particles)
        p[:, 2] = np.random.normal((x1 + x2) / 2.0 + w * 0.5, w * 0.05, self.num_particles)
        p[:, 3] = np.random.normal((y1 + y2) / 2.0 + h * 0.5, h * 0.05, self.num_particles)
        return p

    def predict(self, sigma=3.0):
        # smaller sigma for smoother motion assumptions
        self.particles += np.random.normal(0, sigma, self.particles.shape)

    def update(self, det):
        d = np.linalg.norm(self.particles - det, axis=1)
        # gaussian weight on L2 distance
        self.weights = np.exp(-d**2 / (2 * (30.0**2)))
        self.weights += 1e-300
        self.weights /= np.sum(self.weights)
        self.resample()

    def resample(self):
        idx = np.random.choice(self.num_particles, self.num_particles, p=self.weights)
        self.particles = self.particles[idx]
        self.weights = np.ones(self.num_particles) / self.num_particles

    def estimate(self):
        return np.average(self.particles, weights=self.weights, axis=0)


# ----------------------
# Track wrapper
# ----------------------
class ParticleTrack:
    def __init__(self, bbox, tid, feature=None, timestamp=None,
                 num_particles=50, confirm_hits=3, ema_alpha=0.2):
        self.id = tid
        self.pf = Particle(bbox, num_particles=num_particles)
        self.lost = 0
        self.age = 1
        self.hits = 1  # number of times associated
        self.confirm_hits = confirm_hits
        self.confirmed = (self.hits >= self.confirm_hits)
        self.last_box = bbox
        self.last_seen = timestamp if timestamp is not None else time.time()
        # appearance feature (vector) - if available, keep EMA average
        self.feature = feature.copy() if feature is not None else None
        self.ema_alpha = ema_alpha

    def predict(self):
        self.pf.predict()
        est = self.pf.estimate()
        # estimated box may be in particle coordinate form: [x1,y1,x2,y2]
        self.age += 1
        return est

    def update(self, bbox, feature=None, timestamp=None):
        self.pf.update(bbox)
        self.last_box = bbox
        self.last_seen = timestamp if timestamp is not None else time.time()
        self.lost = 0
        self.hits += 1
        if feature is not None:
            if self.feature is None:
                self.feature = feature.copy()
            else:
                # exponential moving average to adapt appearance slowly
                self.feature = (1.0 - self.ema_alpha) * self.feature + self.ema_alpha * feature
        self.confirmed = (self.hits >= self.confirm_hits)


# ----------------------
# Tracker main class
# ----------------------
class ParticleTracker:
    def __init__(self,
                 max_lost=30,
                 iou_weight=0.5,
                 dist_weight=0.2,
                 app_weight=0.3,
                 distance_decay=80.0,
                 min_confirm_hits=3,
                 reid_memory_time=3.0,
                 num_particles=50):
        """
        max_lost: frames after which a track is considered gone
        iou_weight, dist_weight, app_weight: weights for the combined matching cost (sum=1)
        distance_decay: used to convert center distance -> similarity
        min_confirm_hits: hits required to mark a track 'confirmed'
        reid_memory_time: seconds to keep recently lost tracks in memory for re-id
        """
        self.tracks = {}            # active tracks: id -> ParticleTrack
        self.next_id = 0
        self.max_lost = max_lost
        self.iou_w = iou_weight
        self.dist_w = dist_weight
        self.app_w = app_weight
        assert abs(self.iou_w + self.dist_w + self.app_w - 1.0) < 1e-6, "weights must sum to 1"
        self.distance_decay = distance_decay
        self.min_confirm_hits = min_confirm_hits
        self.reid_memory_time = reid_memory_time
        self.memory = {}            # id -> (last_box, feature, last_seen_time)
        self.num_particles = num_particles

    # ----------------------
    # geometry / similarity helpers
    # ----------------------
    @staticmethod
    def iou(b1, b2):
        x1, y1, x2, y2 = max(b1[0], b2[0]), max(b1[1], b2[1]), min(b1[2], b2[2]), min(b1[3], b2[3])
        w, h = max(0, x2 - x1), max(0, y2 - y1)
        inter = w * h
        area1 = max(0, (b1[2]-b1[0])*(b1[3]-b1[1]))
        area2 = max(0, (b2[2]-b2[0])*(b2[3]-b2[1]))
        union = area1 + area2 - inter
        return inter / union if union > 0 else 0.0

    @staticmethod
    def center_distance(b1, b2):
        c1 = ((b1[0]+b1[2])/2.0, (b1[1]+b1[3])/2.0)
        c2 = ((b2[0]+b2[2])/2.0, (b2[1]+b2[3])/2.0)
        return np.linalg.norm(np.array(c1) - np.array(c2))

    def appearance_similarity(self, f1, f2):
        # returns similarity in [0,1]
        if f1 is None or f2 is None:
            return 0.0
        return clamp(cosine_sim(f1, f2), 0.0, 1.0)

    # ----------------------
    # Matching
    # ----------------------
    def build_cost_matrix(self, track_boxes, track_features, detections, det_features):
        n_t = len(track_boxes)
        n_d = len(detections)
        if n_t == 0 or n_d == 0:
            return np.zeros((n_t, n_d), dtype=np.float32)

        cost = np.zeros((n_t, n_d), dtype=np.float32)
        for i, tb in enumerate(track_boxes):
            for j, db in enumerate(detections):
                iou_score = self.iou(tb, db)          # [0,1]
                dist = self.center_distance(tb, db)
                dist_score = np.exp(-dist / (self.distance_decay + 1e-9))  # [0,1]
                app_score = self.appearance_similarity(track_features[i], det_features[j])  # [0,1]
                # combined similarity
                sim = self.iou_w * iou_score + self.dist_w * dist_score + self.app_w * app_score
                # cost for hungarian is 1 - similarity (lower better)
                cost[i, j] = 1.0 - sim
        return cost

    # ----------------------
    # Public update: detections can be list of bboxes or list of dict {'bbox':..,'feature':..}
    # frame is required if detections are raw bboxes and we want to compute appearance features
    # ----------------------
    def update(self, detections, frame=None, timestamp=None):
        """
        detections: list of bboxes [x1,y1,x2,y2] OR list of dicts {'bbox':..., 'feature':...}
        frame: current video frame (BGR numpy array) - used to compute appearance features if needed
        returns: list of tuples (bbox, track_id, confirmed_bool)
        """
        timestamp = timestamp if timestamp is not None else time.time()

        # normalize detections to (bbox, feature) pairs
        det_bboxes = []
        det_features = []
        for det in detections:
            if isinstance(det, dict):
                bbox = det['bbox']
                feature = det.get('feature', None)
            else:
                bbox = det
                feature = None
            det_bboxes.append(bbox)
            if feature is None:
                # compute simple color histogram if frame is available, else None
                if frame is not None:
                    feature = color_histogram_feature(frame, bbox)
                else:
                    feature = None
            det_features.append(feature)

        # If no active tracks, create new tracks for all detections
        if not self.tracks:
            outputs = []
            for j, bbox in enumerate(det_bboxes):
                fid = self.next_id
                tr = ParticleTrack(bbox, fid, feature=det_features[j],
                                   timestamp=timestamp, num_particles=self.num_particles,
                                   confirm_hits=self.min_confirm_hits)
                tr.confirmed = tr.hits >= self.min_confirm_hits
                self.tracks[fid] = tr
                self.next_id += 1
                outputs.append((bbox, fid, tr.confirmed))
            return outputs

        # Prepare track lists
        track_ids = list(self.tracks.keys())
        track_boxes = []
        track_features = []
        # predict all tracks and get predicted boxes
        for tid in track_ids:
            tr = self.tracks[tid]
            pred = tr.predict()
            # ensure pred is in bbox form and numerical
            pred_box = [float(pred[0]), float(pred[1]), float(pred[2]), float(pred[3])]
            track_boxes.append(pred_box)
            track_features.append(tr.feature)

        # compute assignment cost and solve
        cost = self.build_cost_matrix(track_boxes, track_features, det_bboxes, det_features)
        row, col = [], []
        if cost.size > 0:
            row, col = linear_sum_assignment(cost)

        assigned_tracks = set()
        assigned_dets = set()
        outputs = []

        # apply matches with a similarity threshold
        for r, c in zip(row, col):
            cval = cost[r, c]
            sim = 1.0 - cval
            # threshold: require at least moderate similarity to accept assignment
            if sim >= 0.25:   # you can tune this (0.25 -> very permissive)
                tid = track_ids[r]
                self.tracks[tid].update(det_bboxes[c], feature=det_features[c], timestamp=timestamp)
                assigned_tracks.add(tid)
                assigned_dets.add(c)
                outputs.append((det_bboxes[c], tid, self.tracks[tid].confirmed))

        # unmatched detections -> attempt memory (reid) or create new track
        for j, bbox in enumerate(det_bboxes):
            if j in assigned_dets:
                continue
            reused_id = None
            # try to match with memory using spatial + appearance similarity
            for mid, (mbox, mfeat, mtime) in list(self.memory.items()):
                # ensure memory age not exceeded
                if timestamp - mtime > self.reid_memory_time:
                    del self.memory[mid]
                    continue
                dist = self.center_distance(bbox, mbox)
                if dist < 120:   # spatial cue for potential reid
                    app_sim = 0.0
                    if mfeat is not None and det_features[j] is not None:
                        app_sim = cosine_sim(mfeat, det_features[j])
                    if app_sim > 0.35:
                        reused_id = mid
                        del self.memory[mid]
                        break
            if reused_id is not None:
                # re-create track with same id and previous feature
                tr = ParticleTrack(bbox, reused_id, feature=det_features[j],
                                   timestamp=timestamp, num_particles=self.num_particles,
                                   confirm_hits=self.min_confirm_hits)
                tr.confirmed = tr.hits >= self.min_confirm_hits
                self.tracks[reused_id] = tr
                outputs.append((bbox, reused_id, tr.confirmed))
                assigned_dets.add(j)
                continue

            # otherwise create new track
            fid = self.next_id
            tr = ParticleTrack(bbox, fid, feature=det_features[j],
                               timestamp=timestamp, num_particles=self.num_particles,
                               confirm_hits=self.min_confirm_hits)
            tr.confirmed = tr.hits >= self.min_confirm_hits
            self.tracks[fid] = tr
            self.next_id += 1
            outputs.append((bbox, fid, tr.confirmed))
            assigned_dets.add(j)

        # update lost counters for unmatched tracks
        for tid in list(self.tracks.keys()):
            if tid not in assigned_tracks:
                tr = self.tracks[tid]
                tr.lost += 1
                # save last box & feature to memory for potential re-id with timestamp
                if tr.lost > 0:
                    self.memory[tid] = (tr.last_box, tr.feature, tr.last_seen)
                if tr.lost > self.max_lost:
                    # expire track permanently
                    if tid in self.tracks:
                        del self.tracks[tid]

        return outputs

    # small utility to get all current confirmed tracks
    def get_confirmed_tracks(self):
        return [(tr.last_box, tid) for tid, tr in self.tracks.items() if tr.confirmed]


In [10]:
# Cell 3: Video setup, fps and Tracker init
cap = cv2.VideoCapture(CONFIG["VIDEO_PATH"])
if not cap.isOpened():
    raise RuntimeError(f"Could not open {CONFIG['VIDEO_PATH']}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"[INFO] video fps: {fps:.2f}, size: {width}x{height}, frames: {total_frames}")

# set detect frequency if unset
if CONFIG["DETECT_EVERY_N_FRAMES"] is None:
    CONFIG["DETECT_EVERY_N_FRAMES"] = max(1, int(round(fps)))  # once per second by default

if CONFIG["TEST_MODE"]:
    total_frames = min(CONFIG["TEST_MAX_FRAMES"], total_frames)
    print(f"[INFO] TEST_MODE: limiting to {total_frames} frames")

tracker = ParticleTracker(
    max_lost=30,
    iou_weight=0.5,
    dist_weight=0.2,
    app_weight=0.3,
    distance_decay=80.0,
    min_confirm_hits=3,
    reid_memory_time=CONFIG["REID_MEMORY_TIME"],
    num_particles=50
)
print("[INFO] ParticleTracker initialized")


[INFO] video fps: 29.97, size: 3840x2160, frames: 1377
[INFO] TEST_MODE: limiting to 100 frames
[INFO] ParticleTracker initialized


In [11]:
def bbox_area_intersection_fraction(bbox, frame_w, frame_h):
    """Returns intersection area fraction relative to bbox area."""
    x1, y1, x2, y2 = bbox
    bx1, by1 = max(0, x1), max(0, y1)
    bx2, by2 = min(frame_w, x2), min(frame_h, y2)
    inter_w, inter_h = max(0, bx2 - bx1), max(0, by2 - by1)
    inter_area = inter_w * inter_h
    bbox_area = max(1.0, (x2 - x1) * (y2 - y1))
    return inter_area / bbox_area  # in [0,1]

def yolo_to_bboxes(result):
    try:
        boxes = result[0].boxes.xyxy.cpu().numpy()
    except Exception:
        boxes = result.boxes.xyxy.cpu().numpy()
    bboxes = []
    for b in boxes:
        x1, y1, x2, y2 = map(float, b[:4])
        bboxes.append([x1, y1, x2, y2])
    return bboxes

def fraction_tracks_out_of_frame(tracker, frame_w, frame_h, threshold_inside=0.2):
    confirmed = [tr for tr in tracker.tracks.values() if tr.confirmed]
    if not confirmed:
        return 0.0
    out_count = 0
    for tr in confirmed:
        x1, y1, x2, y2 = tr.last_box
        bx1, by1 = max(0, x1), max(0, y1)
        bx2, by2 = min(frame_w, x2), min(frame_h, y2)
        inter_w, inter_h = max(0, bx2 - bx1), max(0, by2 - by1)
        inter_area = inter_w * inter_h
        bbox_area = max(1.0, (x2 - x1) * (y2 - y1))
        frac = inter_area / bbox_area
        if frac < threshold_inside:
            out_count += 1
    return out_count / len(confirmed)



In [12]:
scene_model.to(CONFIG["DEVICE"])
scene_model.eval()
print("[INFO] ✅ Scene-level TimeSformer model ready for inference.")

# 🎬 Add labels here
scene_labels = [
    "Carrying",
    "Calling",
    "Drinking",
    "Handshaking",
    "Hugging",
    "Lying",
    "Pushing_or_Pulling",
    "Reading",
    "Running",
    "Sitting",
    "Standing",
    "Walking"
]
print(f"[INFO] ✅ Loaded {len(scene_labels)} scene-level action labels.")
print(f"[INFO] Scene-level action labels: {scene_labels}")

[INFO] ✅ Scene-level TimeSformer model ready for inference.
[INFO] ✅ Loaded 12 scene-level action labels.
[INFO] Scene-level action labels: ['Carrying', 'Calling', 'Drinking', 'Handshaking', 'Hugging', 'Lying', 'Pushing_or_Pulling', 'Reading', 'Running', 'Sitting', 'Standing', 'Walking']


In [13]:
person_labels = [
    "Carrying",
    "Calling",
    "Drinking",
    "Handshaking",
    "Hugging",
    "Lying",
    "Pushing_or_Pulling",
    "Reading",
    "Running",
    "Sitting",
    "Standing",
    "Walking"
]
print(f"[INFO] ✅ Loaded {len(person_labels)} person-level action labels.")
print(f"[INFO] Person-level action labels: {person_labels}")

[INFO] ✅ Loaded 12 person-level action labels.
[INFO] Person-level action labels: ['Carrying', 'Calling', 'Drinking', 'Handshaking', 'Hugging', 'Lying', 'Pushing_or_Pulling', 'Reading', 'Running', 'Sitting', 'Standing', 'Walking']


In [14]:
# ============================================================
# 🎯 YOLO + TimeSformer (Scene) + Swin (Person) Inference
# with System / Video Metrics Reporting
# ============================================================
import os, time, cv2, torch, numpy as np, pandas as pd, json
from collections import deque
from tqdm import tqdm
from PIL import Image
from mmaction.apis import inference_recognizer
import statistics

# Optional monitoring libs
try:
    import psutil
except Exception:
    psutil = None
try:
    import GPUtil
except Exception:
    GPUtil = None

# ------------------------------------------------------------
# Compatibility fix for NumPy ≥2.0
# ------------------------------------------------------------
if not hasattr(np, "int"):
    np.int = int

# ============================================================
# ⚙️ CONFIG
# ============================================================
NUM_FRAMES = CONFIG["NUM_FRAMES"]
CONF_THRESH = CONFIG["CONF_THRESH"]
device = CONFIG["DEVICE"]

# 🎛️ FUSION + FILTER OPTIONS (tunable)
CONFIG.setdefault("SWIN_WEIGHT", 0.6)        # 0.0 = only TimeSformer, 1.0 = only Swin
CONFIG.setdefault("TSF_CONF_THRESH", 0.2)
CONFIG.setdefault("SWIN_CONF_THRESH", 0.2)
CONFIG.setdefault("FUSED_CONF_THRESH", 0.3)
CONFIG.setdefault("KEEP_TOP_K", 5)
CONFIG.setdefault("MAX_FINAL_ACTIONS", 2)
CONFIG.setdefault("PERSON_FPS_TMP", 5)

print(f"[INFO] Dual-Stream Inference | SWIN_WEIGHT={CONFIG['SWIN_WEIGHT']} | "
      f"FUSED_CONF={CONFIG['FUSED_CONF_THRESH']} | KEEP_TOP={CONFIG['KEEP_TOP_K']}")

# CONFIG.update({
#     "CONF_THRESH": 0.6,              # less strict detection
#     "DETECT_EVERY_N_FRAMES": 3,      # detect every 3 frames (≈3 Hz)
#     "NUM_FRAMES": 4,                 # short clips for faster updates
#     "TOP_K": 2,
#     "OUT_OF_FRAME_FRACTION": 0.6,
#     "REID_MEMORY_TIME": 10.0,

#     "SWIN_WEIGHT": 0.6,
#     "TSF_CONF_THRESH": 0.2,
#     "SWIN_CONF_THRESH": 0.2,
#     "FUSED_CONF_THRESH": 0.35,
#     "KEEP_TOP_K": 5,
#     "MAX_FINAL_ACTIONS": 2,
#     "PERSON_FPS_TMP": cap.get(cv2.CAP_PROP_FPS),            # match dataset FPS
# })

# CONFIG.update({
#     # ------------------------------------------------------------
#     # 🧠 YOLO + Tracker
#     # ------------------------------------------------------------
#     "CONF_THRESH": 0.6,            # slightly relaxed to catch small UAV subjects
#     "DETECT_EVERY_N_FRAMES": 3,    # detect every ~0.1s (at 30 FPS)
#     "OUT_OF_FRAME_FRACTION": 0.6,  # trigger redetect if >60% go out
#     "REID_MEMORY_TIME": 8.0,       # tracks reconnect for ~8 seconds

#     # ------------------------------------------------------------
#     # 🎬 Temporal Clip Parameters
#     # ------------------------------------------------------------
#     "NUM_FRAMES": 8,               # longer clips for smoother motion at high FPS
#     "TOP_K": 2,                    # top 2 actions retained
#     "PERSON_FPS_TMP": 15,          # half of 30FPS, for Swin temporal scaling

#     # ------------------------------------------------------------
#     # 🧩 Fusion + Confidence Filters
#     # ------------------------------------------------------------
#     "SWIN_WEIGHT": 0.6,            # Swin slightly dominant (per-person)
#     "TSF_CONF_THRESH": 0.25,       # suppress weak TimeSformer logits
#     "SWIN_CONF_THRESH": 0.25,      # suppress weak Swin logits
#     "FUSED_CONF_THRESH": 0.35,     # final acceptance threshold
#     "KEEP_TOP_K": 5,               # keep 5 before filtering
#     "MAX_FINAL_ACTIONS": 2,        # max 2 valid actions per person

#     # ------------------------------------------------------------
#     # ⚙️ System
#     # ------------------------------------------------------------
#     "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
#     "TEST_MODE": True,
#     "TEST_MAX_FRAMES": 500,
# })


[INFO] Dual-Stream Inference | SWIN_WEIGHT=0.6 | FUSED_CONF=0.3 | KEEP_TOP=5


In [15]:
# ============================================================
# 🧩 Setup
# ============================================================
frame_idx = 0
last_detect_frame = -999
frame_buffers = {}        # track_id -> deque(frames)
action_predictions = {}   # track_id -> (labels, scores)
results_rows = []

# Labels for Okutama dataset
person_labels = [
    "Calling","Carrying","Drinking","Handshaking","Hugging","Lying",
    "Pushing_or_Pulling","Reading","Running","Sitting","Standing","Walking"
]
num_classes = len(person_labels)
label_to_idx = {l:i for i,l in enumerate(person_labels)}

VIDEO_PATH   = CONFIG["VIDEO_PATH"]
VIDEO_NAME   = CONFIG.get("VIDEO_NAME", os.path.splitext(os.path.basename(VIDEO_PATH))[0])
OUTPUT_DIR   = CONFIG["OUTPUT_DIR"]
RESULTS_CSV  = CONFIG["RESULTS_CSV"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

video_fps = cap.get(cv2.CAP_PROP_FPS) or None
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if CONFIG.get("TEST_MODE", False):
    total_frames = min(CONFIG["TEST_MAX_FRAMES"], total_frames)

print(f"[INFO] 🚀 Running fusion on {total_frames} frames... (video FPS: {video_fps})")

# ============================================================
# 🔧 Helper Functions (same as before)
# ============================================================
def as_pil_list(frames_bgr):
    return [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frames_bgr]

def run_timesformer_on_clip(frames_bgr):
    pil_frames = as_pil_list(frames_bgr)
    inputs = processor_scene(pil_frames, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = scene_model(**inputs).logits
        probs = torch.sigmoid(logits)[0].detach().cpu().numpy()
    probs = np.asarray(probs, dtype=np.float32)
    probs = np.clip(probs, 0, 1)
    probs[probs < CONFIG["TSF_CONF_THRESH"]] = 0
    return probs

import tempfile
def run_swin_on_clip(frames_bgr):
    h, w = frames_bgr[0].shape[:2]
    # use system temp dir to avoid cluttering OUTPUT_DIR
    tmp_path = os.path.join(tempfile.gettempdir(), f"_swin_{int(time.time()*1000)}.mp4")
    vw = cv2.VideoWriter(tmp_path, cv2.VideoWriter_fourcc(*'mp4v'),
                         CONFIG["PERSON_FPS_TMP"], (w, h))
    for f in frames_bgr:
        vw.write(f)
    vw.release()

    vec = np.zeros(num_classes, dtype=np.float32)
    try:
        results = inference_recognizer(swin_model, tmp_path)
        for label, score in results:
            if isinstance(label, int):
                if 0 <= label < num_classes:
                    vec[label] = max(vec[label], float(score))
            else:
                idx = label_to_idx.get(str(label))
                if idx is not None:
                    vec[idx] = max(vec[idx], float(score))
    finally:
        try: os.remove(tmp_path)
        except: pass

    vec[vec < CONFIG["SWIN_CONF_THRESH"]] = 0
    return vec

def fuse_probs(swin_p, tsf_p, alpha):
    fused = alpha * swin_p + (1.0 - alpha) * tsf_p
    fused[fused < CONFIG["FUSED_CONF_THRESH"]] = 0
    return fused

# ============================================================
# 🚫 Conflict Filtering (same logic)
# ============================================================
POSTURE_GROUP = {"Standing","Sitting","Lying"}
MOTION_GROUP = {"Running","Walking"}
INTERACTION_GROUP = {"Hugging","Handshaking","Pushing_or_Pulling"}

def are_conflicting(a, b):
    groups = [POSTURE_GROUP, MOTION_GROUP, INTERACTION_GROUP]
    for g in groups:
        if a in g and b in g:
            return True
    if (a in POSTURE_GROUP and b in MOTION_GROUP) or (b in POSTURE_GROUP and a in MOTION_GROUP):
        return True
    return False

def clean_action_combos(actions, scores, max_out=2):
    paired = list(zip(actions, scores))
    to_remove = set()
    for i in range(len(actions)):
        for j in range(i + 1, len(actions)):
            a, b = actions[i], actions[j]
            if are_conflicting(a, b):
                if scores[i] >= scores[j]:
                    to_remove.add(b)
                else:
                    to_remove.add(a)
    filtered = [(a, s) for a, s in paired if a not in to_remove and s > 0]
    filtered.sort(key=lambda x: x[1], reverse=True)
    final = filtered[:max_out] if filtered else paired[:max_out]
    acts, scrs = zip(*final)
    return list(acts), list(scrs)

# ============================================================
# 📊 Metrics collection structures
# ============================================================
metric_samples = {
    "frame_times": [],           # seconds per frame processing
    "cpu_percent": [],           # sampled %
    "system_ram_percent": [],    # system memory %
    "proc_rss_gb": [],           # process memory RSS (GB)
    "gpu_mem_gb": [],            # torch.cuda.memory_allocated (GB)
    "gpu_mem_peak_gb": [],       # torch.cuda.max_memory_allocated (GB)
    "gpu_util_percent": [],      # via GPUtil if available
}

# reset torch peak stats if cuda available
if torch.cuda.is_available():
    try:
        torch.cuda.reset_peak_memory_stats()
    except Exception:
        pass

# get psutil process handle if available
proc = psutil.Process() if psutil is not None else None

def sample_system_metrics():
    """Sample current CPU/RAM/GPU/process memory and append to lists."""
    # CPU %
    try:
        if psutil is not None:
            cpu_p = psutil.cpu_percent(interval=None)
            ram_p = psutil.virtual_memory().percent
            rss = proc.memory_info().rss / (1024 ** 3)  # GB
        else:
            cpu_p, ram_p, rss = None, None, None
    except Exception:
        cpu_p, ram_p, rss = None, None, None

    # GPU mem & peak
    gpu_mem, gpu_peak, gpu_util = None, None, None
    try:
        if torch.cuda.is_available():
            dev = torch.cuda.current_device()
            gpu_mem = torch.cuda.memory_allocated(dev) / (1024 ** 3)
            try:
                gpu_peak = torch.cuda.max_memory_allocated(dev) / (1024 ** 3)
            except Exception:
                gpu_peak = None
        if GPUtil is not None:
            gpus = GPUtil.getGPUs()
            if len(gpus) > 0:
                gpu_util = float(gpus[0].load) * 100.0
                # if GPUtil provides memoryUsed in MB
                if gpu_mem is None:
                    try:
                        gpu_mem = gpus[0].memoryUsed / 1024.0
                    except Exception:
                        pass
    except Exception:
        gpu_mem, gpu_peak, gpu_util = None, None, None

    # append
    metric_samples["cpu_percent"].append(cpu_p)
    metric_samples["system_ram_percent"].append(ram_p)
    metric_samples["proc_rss_gb"].append(rss)
    metric_samples["gpu_mem_gb"].append(gpu_mem)
    metric_samples["gpu_mem_peak_gb"].append(gpu_peak)
    metric_samples["gpu_util_percent"].append(gpu_util)

# ============================================================
# 🎥 Main Loop (with sampling)
# ============================================================
start_time = time.time()
last_sample_time = start_time

while frame_idx < total_frames:
    t0 = time.time()
    ret, frame = cap.read()
    if not ret:
        break

    do_detect = (frame_idx - last_detect_frame) >= CONFIG["DETECT_EVERY_N_FRAMES"] or len(tracker.tracks) == 0
    frac_out = fraction_tracks_out_of_frame(tracker, W, H, threshold_inside=0.2)
    if frac_out >= 0.6:
        do_detect = True

    if do_detect:
        yolo_res = yolo_model.predict(frame, conf=CONF_THRESH, verbose=False)
        dets = yolo_to_bboxes(yolo_res)
        last_detect_frame = frame_idx
        tracker.update(dets, frame=frame, timestamp=time.time())
    else:
        tracker.update([], frame=frame, timestamp=time.time())

    for tid, tr in list(tracker.tracks.items()):
        if tr.lost > tracker.max_lost:
            frame_buffers.pop(tid, None)
            action_predictions.pop(tid, None)

    for tid, tr in list(tracker.tracks.items()):
        if not tr.confirmed:
            continue
        x1, y1, x2, y2 = map(int, tr.last_box)
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
        if x2 <= x1 or y2 <= y1:
            continue

        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        frame_buffers.setdefault(tid, deque(maxlen=NUM_FRAMES)).append(crop)

        if len(frame_buffers[tid]) == NUM_FRAMES and (frame_idx % NUM_FRAMES == 0):
            clip = list(frame_buffers[tid])
            tsf_probs = run_timesformer_on_clip(clip)
            swin_probs = run_swin_on_clip(clip)
            fused = fuse_probs(swin_probs, tsf_probs, CONFIG["SWIN_WEIGHT"])

            top_idx = np.argsort(fused)[::-1][:CONFIG["KEEP_TOP_K"]]
            top_actions = [person_labels[i] for i in top_idx]
            top_scores = [float(fused[i]) for i in top_idx]

            final_actions, final_scores = clean_action_combos(
                top_actions, top_scores, CONFIG["MAX_FINAL_ACTIONS"]
            )

            action_predictions[tid] = (final_actions, final_scores)
            print(f"[PERSON] Frame {frame_idx} | ID {tid} -> {list(zip(final_actions, [f'{s:.3f}' for s in final_scores]))}")

        acts, scores = action_predictions.get(tid, ([], []))
        results_rows.append((frame_idx, tid, x1, y1, x2, y2, int(tr.confirmed),
                             ",".join(acts), ",".join([f"{s:.3f}" for s in scores])))

    frame_idx += 1

    t1 = time.time()
    frame_time = t1 - t0
    metric_samples["frame_times"].append(frame_time)

    # sample system metrics every frame (cheap) or you may sample every N frames
    sample_system_metrics()

# ============================================================
# 💾 Save Results (existing)
# ============================================================
cap.release()
elapsed = time.time() - start_time
print(f"[INFO] ✅ Done {frame_idx} frames in {elapsed:.2f}s ({frame_idx/elapsed:.2f} FPS)")

df = pd.DataFrame(results_rows, columns=[
    "frame_idx","track_id","x1","y1","x2","y2","confirmed","actions","scores"
])
df = df.drop_duplicates(subset=["frame_idx","track_id"], keep="last")
df.to_csv(RESULTS_CSV, index=False)
print(f"✅ Saved: {RESULTS_CSV}")

# ============================================================
# 📈 Summarize & Save Metrics
# ============================================================
def safe_stats(lst):
    lstf = [x for x in lst if x is not None]
    if not lstf:
        return {"avg": None, "min": None, "max": None}
    return {"avg": float(statistics.mean(lstf)), "min": float(min(lstf)), "max": float(max(lstf))}

metrics_summary = {
    "video_name": VIDEO_NAME,
    "video_path": VIDEO_PATH,
    "video_fps_reported": video_fps,
    "frames_processed": frame_idx,
    "total_time_s": elapsed,
    "effective_fps": frame_idx / elapsed if elapsed > 0 else None,
    "frame_time_stats_s": safe_stats(metric_samples["frame_times"]),
    "cpu_percent_stats": safe_stats(metric_samples["cpu_percent"]),
    "system_ram_percent_stats": safe_stats(metric_samples["system_ram_percent"]),
    "process_rss_gb_stats": safe_stats(metric_samples["proc_rss_gb"]),
    "gpu_mem_gb_stats": safe_stats(metric_samples["gpu_mem_gb"]),
    "gpu_mem_peak_gb_stats": safe_stats(metric_samples["gpu_mem_peak_gb"]),
    "gpu_util_percent_stats": safe_stats(metric_samples["gpu_util_percent"]),
    "notes": {
        "psutil_available": psutil is not None,
        "GPUtil_available": GPUtil is not None,
        "torch_cuda_available": torch.cuda.is_available()
    }
}

# pretty print
print("\n[METRICS SUMMARY]")
print(json.dumps(metrics_summary, indent=2))

# save json & csv
metrics_json_path = os.path.join(OUTPUT_DIR, f"{VIDEO_NAME}_metrics.json")
with open(metrics_json_path, "w") as f:
    json.dump(metrics_summary, f, indent=2)
print(f"[METRICS] Saved JSON → {metrics_json_path}")

# also save a CSV with per-sample aggregated stats (optional)
metrics_rows = []
n_samples = len(metric_samples["frame_times"])
for i in range(n_samples):
    metrics_rows.append({
        "sample_idx": i,
        "frame_time_s": metric_samples["frame_times"][i] if i < len(metric_samples["frame_times"]) else None,
        "cpu_percent": metric_samples["cpu_percent"][i] if i < len(metric_samples["cpu_percent"]) else None,
        "system_ram_percent": metric_samples["system_ram_percent"][i] if i < len(metric_samples["system_ram_percent"]) else None,
        "proc_rss_gb": metric_samples["proc_rss_gb"][i] if i < len(metric_samples["proc_rss_gb"]) else None,
        "gpu_mem_gb": metric_samples["gpu_mem_gb"][i] if i < len(metric_samples["gpu_mem_gb"]) else None,
        "gpu_util_percent": metric_samples["gpu_util_percent"][i] if i < len(metric_samples["gpu_util_percent"]) else None,
    })

metrics_df = pd.DataFrame(metrics_rows)
metrics_csv_path = os.path.join(OUTPUT_DIR, f"{VIDEO_NAME}_metrics_samples.csv")
metrics_df.to_csv(metrics_csv_path, index=False)
print(f"[METRICS] Saved samples CSV → {metrics_csv_path}")

print("[INFO] All done.")


[INFO] 🚀 Running fusion on 100 frames... (video FPS: 29.97002997002997)


[ WARN:0@8.629] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@8.629] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@8.629] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@8.731] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@8.731] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@8.731] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame


[PERSON] Frame 8 | ID 0 -> [('Pushing_or_Pulling', '0.454'), ('Sitting', '0.397')]
[PERSON] Frame 8 | ID 1 -> [('Walking', '0.472')]


[ WARN:0@8.912] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@8.912] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@8.912] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@9.002] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@9.002] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@9.002] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame


[PERSON] Frame 12 | ID 0 -> [('Sitting', '0.604')]
[PERSON] Frame 12 | ID 1 -> [('Sitting', '0.330')]


...

[PERSON] Frame 92 | ID 0 -> [('Sitting', '0.361')]
[PERSON] Frame 92 | ID 1 -> [('Walking', '0.354')]
[PERSON] Frame 92 | ID 2 -> [('Sitting', '0.576')]


[ WARN:0@16.179] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@16.179] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@16.179] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@16.354] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@16.354] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame


[PERSON] Frame 92 | ID 3 -> [('Standing', '0.376')]
[PERSON] Frame 96 | ID 0 -> [('Standing', '0.513'), ('Pushing_or_Pulling', '0.353')]


[ WARN:0@16.443] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@16.444] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@16.533] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@16.533] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@16.533] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@16.623] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@16.623] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame
[ WARN:0@16.623] global cap_ffmpeg.cpp:198 write FFmpeg: Failed to write frame


[PERSON] Frame 96 | ID 1 -> [('Walking', '0.420')]
[PERSON] Frame 96 | ID 2 -> [('Sitting', '0.600')]
[PERSON] Frame 96 | ID 3 -> [('Standing', '0.582')]
[INFO] ✅ Done 100 frames in 8.85s (11.30 FPS)
✅ Saved: /home1/jtt_1/mtp/outputs/2.2.1/2.2.1_tracking.csv

[METRICS SUMMARY]
{
  "video_name": "2.2.1",
  "video_path": "/home1/jtt_1/mtp/okutama-action/TestSetVideos/Drone2/Noon/2.2.1.mp4",
  "video_fps_reported": 29.97002997002997,
  "frames_processed": 100,
  "total_time_s": 8.850656270980835,
  "effective_fps": 11.29859718175655,
  "frame_time_stats_s": {
    "avg": 0.07518409729003907,
    "min": 0.008120298385620117,
    "max": 0.4097268581390381
  },
  "cpu_percent_stats": {
    "avg": 8.496,
    "min": 2.9,
    "max": 24.7
  },
  "system_ram_percent_stats": {
    "avg": 18.938,
    "min": 18.4,
    "max": 19.2
  },
  "process_rss_gb_stats": {
    "avg": 1.9213267135620118,
    "min": 1.7517662048339844,
    "max": 1.9602394104003906
  },
  "gpu_mem_gb_stats": {
    "avg": 1.085535

In [16]:
import psutil, time
from collections import deque

try:
    import GPUtil
except Exception:
    GPUtil = None


In [17]:
# ============================================================
# 🎬 DUAL-VIEW VISUALIZATION: Person-Level (Swin + TimeSformer Fused)
# ============================================================
import cv2
import pandas as pd
import os
import random
from tqdm import tqdm

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------
VIDEO_PATH   = CONFIG["VIDEO_PATH"]
VIDEO_NAME   = CONFIG["VIDEO_NAME"]
OUTPUT_DIR   = CONFIG["OUTPUT_DIR"]
CSV_PATH     = CONFIG["RESULTS_CSV"]
OUTPUT_VIDEO = os.path.join(OUTPUT_DIR, f"{VIDEO_NAME}_fused_vis.mp4")

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"❌ Results CSV not found: {CSV_PATH}")

# ------------------------------------------------------------
# LOAD VIDEO + RESULTS
# ------------------------------------------------------------
df = pd.read_csv(CSV_PATH)
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"❌ Cannot open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

try:
    out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (w, h))
except Exception:
    print("⚠️ FFmpeg not available — skipping video writer.")
    out = None

total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if CONFIG.get("TEST_MODE", False):
    total = min(CONFIG.get("TEST_MAX_FRAMES", 300), total)

print(f"\n[INFO] 🎞 Rendering {total} frames ({w}x{h}, {fps:.2f} FPS)")
print(f"[INFO] Output → {OUTPUT_VIDEO}")

# ------------------------------------------------------------
# LABELS
# ------------------------------------------------------------
person_labels = [
    "Calling", "Carrying", "Drinking", "Handshaking", "Hugging",
    "Lying", "Pushing_or_Pulling", "Reading", "Running",
    "Sitting", "Standing", "Walking"
]

def decode_actions(raw_str):
    """Convert numeric or comma-separated string to readable action list."""
    if not isinstance(raw_str, str) or not raw_str.strip():
        return []
    parts = [p.strip() for p in raw_str.split(",") if p.strip()]
    decoded = []
    for p in parts:
        if p.isdigit() and int(p) < len(person_labels):
            decoded.append(person_labels[int(p)])
        else:
            decoded.append(p)
    return decoded

# ------------------------------------------------------------
# VISUAL STYLE
# ------------------------------------------------------------
FONT_SCALE = max(0.6, min(1.5, h / 720 * 0.8))
THICKNESS  = int(max(2, h / 720 * 2))
PADDING_X, PADDING_Y = int(10 * (h / 720)), int(6 * (h / 720))

# ------------------------------------------------------------
# COLOR HELPERS
# ------------------------------------------------------------
track_colors = {}
def get_color(tid):
    if tid not in track_colors:
        random.seed(int(tid) + 42)
        track_colors[tid] = tuple(random.randint(50, 230) for _ in range(3))
    return track_colors[tid]

def lighten(c, f=1.4): 
    return tuple(min(255, int(x * f)) for x in c)

# ------------------------------------------------------------
# DRAW HELPERS
# ------------------------------------------------------------
def draw_filled_rounded_rect(img, x1, y1, x2, y2, color, radius=8, alpha=0.9):
    overlay = img.copy()
    cv2.rectangle(overlay, (x1 + radius, y1), (x2 - radius, y2), color, -1)
    cv2.rectangle(overlay, (x1, y1 + radius), (x2, y2 - radius), color, -1)
    for cx, cy in [(x1 + radius, y1 + radius), (x2 - radius, y1 + radius),
                   (x1 + radius, y2 - radius), (x2 - radius, y2 - radius)]:
        cv2.circle(overlay, (cx, cy), radius, color, -1)
    cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0, img)

def draw_label(frame, text, pos, color, alpha=0.85):
    """Draw pretty rounded label box."""
    if not isinstance(text, str):
        text = str(text)
    font = cv2.FONT_HERSHEY_SIMPLEX
    x, y = pos
    (tw, th), _ = cv2.getTextSize(text, font, FONT_SCALE, THICKNESS)
    x1, y1 = max(0, x - PADDING_X), max(0, y - th - PADDING_Y)
    x2, y2 = min(frame.shape[1]-1, x + tw + PADDING_X), min(frame.shape[0]-1, y + PADDING_Y)
    draw_filled_rounded_rect(frame, x1, y1, x2, y2, lighten(color, 1.5), radius=8, alpha=alpha)
    cv2.putText(frame, text, (x + 3, y - 2), font, FONT_SCALE, (0,0,0), THICKNESS + 2, cv2.LINE_AA)
    cv2.putText(frame, text, (x + 3, y - 2), font, FONT_SCALE, (255,255,255), THICKNESS, cv2.LINE_AA)

# ------------------------------------------------------------
# ADD HEADER (MODEL TITLE)
# ------------------------------------------------------------
def draw_header_banner(frame):
    title = "YOLO + Swin Video Transfomer + TimeSformer (Fused Multi-Label Action Recognition)"
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), _ = cv2.getTextSize(title, font, FONT_SCALE * 0.9, THICKNESS)
    pad = 15
    banner_color = (30, 30, 30)
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, th + 3 * pad), banner_color, -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    cv2.putText(frame, title, (pad, th + pad), font, FONT_SCALE * 0.9, (255, 255, 255), THICKNESS, cv2.LINE_AA)

# ------------------------------------------------------------
# RENDER LOOP
# ------------------------------------------------------------
frame_groups = df.groupby("frame_idx")
pbar = tqdm(total=total, desc="[Rendering Video]", unit="frame")

for i in range(total):
    ret, frame = cap.read()
    if not ret:
        break

    # Header
    draw_header_banner(frame)

    # --- Person-level annotation ---
    if i in frame_groups.groups:
        for _, r in frame_groups.get_group(i).iterrows():
            tid = int(r.track_id)
            x1, y1, x2, y2 = map(int, [r.x1, r.y1, r.x2, r.y2])
            color = get_color(tid)

            acts = decode_actions(r.get("actions", ""))
            act_text = " | ".join(acts) if acts else ""

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, THICKNESS)
            draw_label(frame, f"ID {tid}", (x1, y1 - 10), color)

            if act_text:
                label_y = y2 + int(35 * (h / 720))
                if label_y + 25 > h:
                    label_y = max(30, y1 - int(45 * (h / 720)))
                draw_label(frame, act_text, (x1, label_y), color)

    if out:
        out.write(frame)
    pbar.update(1)

pbar.close()
cap.release()
if out:
    out.release()

print(f"✅ Visualization saved → {OUTPUT_VIDEO}")



[INFO] 🎞 Rendering 100 frames (3840x2160, 29.97 FPS)
[INFO] Output → /home1/jtt_1/mtp/outputs/2.2.1/2.2.1_fused_vis.mp4


[Rendering Video]: 100%|██████████| 100/100 [00:07<00:00, 13.49frame/s]

✅ Visualization saved → /home1/jtt_1/mtp/outputs/2.2.1/2.2.1_fused_vis.mp4


In [18]:
# import os
# import pandas as pd

# # ============================================================
# # CONFIGURATION (yours)
# # ============================================================
# CONFIG = {
#     "VIDEO_PATH": "/home1/jtt_1/mtp/timesformer-dataset-test/Drone1/Morning/1.1.8.mp4",
#     "RESULTS_CSV": "/home1/jtt_1/mtp/outputs/1.1.8/1.1.8_tracking.csv",
#     "REID_MEMORY_TIME": 10.0,
# }

# # ============================================================
# # 1️⃣ Extract Video Name and Folder
# # ============================================================
# video_name = os.path.splitext(os.path.basename(CONFIG["VIDEO_PATH"]))[0]
# video_dir  = os.path.dirname(CONFIG["VIDEO_PATH"])
# results_csv = CONFIG["RESULTS_CSV"]

# print(f"\n🎞 Video name: {video_name}")
# print(f"📂 Video directory: {video_dir}")
# print(f"📄 Tracking CSV: {results_csv}")

# # ============================================================
# # 2️⃣ Auto-Find Matching BBox CSV
# # ============================================================
# bbox_csv = None
# for f in os.listdir(video_dir):
#     if f.startswith(video_name) and f.endswith("_bboxes.csv"):
#         bbox_csv = os.path.join(video_dir, f)
#         break

# if bbox_csv is None:
#     raise FileNotFoundError(f"No bbox CSV found for {video_name} in {video_dir}")

# print(f"✅ Found bbox CSV: {bbox_csv}")

# # ============================================================
# # 3️⃣ Load Both CSVs
# # ============================================================
# df_bboxes = pd.read_csv(bbox_csv)
# df_track  = pd.read_csv(results_csv)

# print(f"📊 BBoxes shape: {df_bboxes.shape}")
# print(f"📊 Tracking shape: {df_track.shape}")

# # ============================================================
# # 4️⃣ Compare Frame Coverage
# # ============================================================
# bbox_frames = df_bboxes["frame"].nunique() if "frame" in df_bboxes.columns else len(df_bboxes)
# track_frames = df_track["frame"].nunique() if "frame" in df_track.columns else len(df_track)

# missing_in_track = set(df_bboxes["frame"]) - set(df_track["frame"])
# missing_in_bbox  = set(df_track["frame"]) - set(df_bboxes["frame"])

# print(f"\n🧩 Frames in bbox CSV: {bbox_frames}")
# print(f"🧩 Frames in tracking CSV: {track_frames}")
# print(f"⚠️ Missing in tracking CSV: {len(missing_in_track)} frames")
# print(f"⚠️ Missing in bbox CSV: {len(missing_in_bbox)} frames")

# if len(missing_in_track) > 0:
#     print(f"   e.g. first 10 missing: {sorted(list(missing_in_track))[:10]}")

# # ============================================================
# # 5️⃣ Save Summary
# # ============================================================
# summary_dir = os.path.dirname(results_csv)
# summary_path = os.path.join(summary_dir, f"{video_name}_compare_summary.csv")

# summary = {
#     "video_name": [video_name],
#     "bbox_csv": [bbox_csv],
#     "tracking_csv": [results_csv],
#     "bbox_frames": [bbox_frames],
#     "track_frames": [track_frames],
#     "missing_in_track": [len(missing_in_track)],
#     "missing_in_bbox": [len(missing_in_bbox)],
# }

# pd.DataFrame(summary).to_csv(summary_path, index=False)
# print(f"\n💾 Saved summary → {summary_path}")


In [19]:
# import cv2
# import statistics
# import torch

# # ============================================================
# # 🎯 AUTO PARAMETER ADVISOR
# # ============================================================
# def auto_tune_parameters(video_path, yolo_model_path, sample_frames=100):
#     """
#     Analyzes a video to recommend YOLO + tracker + action classifier parameters.
#     """

#     print(f"[INFO] Analyzing first {sample_frames} frames from: {os.path.basename(video_path)}")

#     cap = cv2.VideoCapture(video_path)
#     if not cap.isOpened():
#         raise RuntimeError(f"Cannot open video: {video_path}")

#     fps = cap.get(cv2.CAP_PROP_FPS)
#     w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#     h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#     total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
#     print(f"[INFO] Resolution: {w}x{h}, FPS: {fps:.2f}, Total Frames: {total_frames}")

#     from ultralytics import YOLO
#     yolo = YOLO(yolo_model_path)
#     conf_candidates = [0.25, 0.5, 0.7]

#     frame_indices = np.linspace(0, min(sample_frames, total_frames - 1), min(sample_frames, total_frames), dtype=int)
#     motion_scores, object_counts = [], []
#     last_gray = None

#     for idx in frame_indices:
#         cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
#         ret, frame = cap.read()
#         if not ret:
#             continue

#         # Compute motion (frame difference)
#         gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#         if last_gray is not None:
#             diff = cv2.absdiff(gray, last_gray)
#             motion_scores.append(np.mean(diff))
#         last_gray = gray

#         # Run quick detection
#         res = yolo.predict(frame, conf=0.5, verbose=False)
#         count = len(res[0].boxes)
#         object_counts.append(count)

#     cap.release()

#     avg_motion = np.mean(motion_scores)
#     motion_std = np.std(motion_scores)
#     avg_objects = np.mean(object_counts)
#     obj_std = np.std(object_counts)

#     print(f"\n[INFO] Scene motion intensity: {avg_motion:.2f} ± {motion_std:.2f}")
#     print(f"[INFO] Avg detected objects/frame: {avg_objects:.2f} ± {obj_std:.2f}")

#     # ------------------------------------------------------------
#     # 🧠 Rule-based suggestions
#     # ------------------------------------------------------------
#     # Confidence threshold
#     if avg_objects > 20:
#         conf = 0.6  # crowded
#     elif avg_objects > 5:
#         conf = 0.5
#     else:
#         conf = 0.7  # sparse scenes prefer higher confidence

#     # Detection frequency
#     if avg_motion > 12:
#         detect_every = 1  # high motion → frequent detection
#     elif avg_motion > 6:
#         detect_every = 3
#     else:
#         detect_every = 5  # low motion → rely more on tracker

#     # Out-of-frame fraction
#     out_frac = 0.8 if avg_motion > 8 else 0.9

#     # Re-ID memory
#     reid_time = 2.0 if avg_motion > 10 else 4.0

#     # TimeSformer clip length
#     if fps >= 30:
#         num_frames = 8
#     elif fps >= 20:
#         num_frames = 12
#     else:
#         num_frames = 16

#     # Top-K actions
#     top_k = 2 if num_frames <= 12 else 3

#     # ------------------------------------------------------------
#     print("\n⚙️  Recommended Parameters")
#     print("="*50)
#     print(f"CONF_THRESH           = {conf}")
#     print(f"DETECT_EVERY_N_FRAMES = {detect_every}")
#     print(f"OUT_OF_FRAME_FRACTION = {out_frac}")
#     print(f"REID_MEMORY_TIME      = {reid_time}")
#     print(f"NUM_FRAMES (TimeSformer clip) = {num_frames}")
#     print(f"TOP_K (actions)       = {top_k}")
#     print("="*50)

#     suggested = {
#         "CONF_THRESH": conf,
#         "DETECT_EVERY_N_FRAMES": detect_every,
#         "OUT_OF_FRAME_FRACTION": out_frac,
#         "REID_MEMORY_TIME": reid_time,
#         "NUM_FRAMES": num_frames,
#         "TOP_K": top_k
#     }
#     return suggested


# # ============================================================
# # 🧪 Run on your current video
# # ============================================================
# suggested_params = auto_tune_parameters(CONFIG["VIDEO_PATH"], CONFIG["YOLO_MODEL_PATH"])

# # You can update CONFIG automatically:
# CONFIG.update(suggested_params)
# print("\n[✅ CONFIG UPDATED WITH RECOMMENDED VALUES]")
